# Pregunta a la línea · Mistral Small 3.2 24B

Prueba de Mistral Small 3.2 24B Instruct (2506) en dos T4 de Kaggle. Los datos limpios se
leen de un dataset privado. Las cifras se calculan en Python y el modelo redacta a partir
de los resultados de las herramientas.

## Configurar y localizar las entradas

Selecciona **GPU T4 ×2**, habilita Internet para instalar el motor y añade los datasets
privados de datos y pesos si no aparecen ya como entradas.

In [ ]:
import hashlib
from pathlib import Path
import os
import secrets
import subprocess
import sys
import tarfile
import threading
import time
from zipfile import ZipFile

import pandas as pd
import requests

try:
    from openai import OpenAI
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "openai==2.9.0"], check=True)
    from openai import OpenAI

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RUNTIME = Path("/tmp/horse-lossia-llama")
RUNTIME.mkdir(exist_ok=True)
MODEL_FILE = "Mistral-Small-3.2-24B-Instruct-2506-Q8_0.gguf"
MODEL_SHA256 = "b2a40e5ef4eab8837d0462c303e8147ec754e2963e41916b551107d2b0ca6527"
MODEL_SIZE = 25054793248
LLM_MODEL = "mistral-small-3.2-24b-instruct-2506-q8_0"
LLM_BASE_URL = "http://127.0.0.1:8080/v1"
RELEASE = "b12366"

def entrada(nombre):
    rutas = sorted(INPUT.rglob(nombre))
    if len(rutas) != 1:
        raise RuntimeError(f"Se esperaba una entrada {nombre} y se encontraron {len(rutas)}.")
    return rutas[0]

# Kaggle puede montar el ZIP o publicar su contenido ya descomprimido.
archivos = list(INPUT.rglob("horse-lossia.zip"))
if archivos:
    repo_dir = RUNTIME / "horse-lossia"
    with ZipFile(entrada("horse-lossia.zip")) as archivo:
        archivo.extractall(repo_dir)
else:
    repo_dir = entrada("interfaz.py").parent.parent
if not (repo_dir / "lossia" / "interfaz.py").is_file():
    raise RuntimeError("El dataset de LOSSIA debe conservar las carpetas del repositorio.")
datos_dir = repo_dir / "Datos_limpios"
resultados_dir = repo_dir / "Resultados"
pesos = entrada(MODEL_FILE)
if pesos.stat().st_size != MODEL_SIZE:
    raise RuntimeError("El tamaño de los pesos de Kaggle difiere del original.")
with pesos.open("rb") as archivo:
    if hashlib.file_digest(archivo, "sha256").hexdigest() != MODEL_SHA256:
        raise RuntimeError("El SHA-256 de los pesos de Kaggle difiere del original.")
print(f"Pesos verificados: {MODEL_SIZE} bytes, SHA-256 {MODEL_SHA256}", flush=True)

def gpu_info():
    salida = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader,nounits"],
        text=True,
    )
    return [(nombre.strip(), int(total), int(usada))
            for nombre, total, usada in (linea.split(",") for linea in salida.splitlines())]

gpu_inicial = gpu_info()
if len(gpu_inicial) != 2 or any("T4" not in gpu[0] for gpu in gpu_inicial):
    raise RuntimeError(f"Selecciona GPU T4 ×2. Detectadas: {gpu_inicial}")
print("GPU:", gpu_inicial)


## Instalar llama.cpp con CUDA

In [ ]:
paquetes = {
    "llama-b12366-bin-ubuntu-cuda-12.8-x64.tar.gz": "7a8d47aab18101b536fc59c7b6c257c1c3eb0e573bf90ecb493fafa2aacc7531",
    "cudart-llama-b12366-bin-ubuntu-cuda-12.8-x64.tar.gz": "32b1f9caafd3de2051b9037d978ef021cc9377c77b3ef7a1d1dac3e6fd55fc4a",
}
install = RUNTIME / RELEASE
install.mkdir(exist_ok=True)
for nombre, sha in paquetes.items():
    descarga = RUNTIME / nombre
    url = f"https://github.com/ggml-org/llama.cpp/releases/download/{RELEASE}/{nombre}"
    if not descarga.is_file():
        parcial = descarga.with_suffix(".part")
        subprocess.run(["curl", "-fsSL", "--connect-timeout", "30", "-o", str(parcial), url], check=True)
        parcial.replace(descarga)
    with descarga.open("rb") as archivo:
        if hashlib.file_digest(archivo, "sha256").hexdigest() != sha:
            raise RuntimeError(f"SHA-256 inesperado para {nombre}")
    with tarfile.open(descarga, "r:gz") as archivo:
        archivo.extractall(install, filter="data")

servidor, = install.rglob("llama-server")
bibliotecas = sorted({str(ruta.parent) for ruta in install.rglob("*.so*")})
entorno = os.environ.copy()
entorno["LD_LIBRARY_PATH"] = os.pathsep.join(bibliotecas + [entorno.get("LD_LIBRARY_PATH", "")])
subprocess.run([str(servidor), "--version"], env=entorno, check=True)


## Herramientas de la línea

Se importan de `chat/herramientas.py`, el mismo módulo que usa Streamlit. Las cifras salen
del motor oficial (`lossia/`), así que coinciden con las del dashboard.


In [ ]:
sys.path.insert(0, str(repo_dir))
from chat.herramientas import (HerramientasLinea, TOOLS, firma_llamadas, preguntar,
                               verificar_cifras)
import json
import re

print([t['function']['name'] for t in TOOLS])

ruta_chat = repo_dir / "chat" / "herramientas.py"
print("SHA-256 de herramientas.py:", hashlib.sha256(ruta_chat.read_bytes()).hexdigest())

## Servir el modelo y comprobar el endpoint

In [ ]:
herramientas = HerramientasLinea(datos_dir, resultados_dir)
if "proceso" in globals() and proceso.poll() is None:
    proceso.terminate()
    proceso.wait(timeout=20)

clave = os.environ.get("LLM_API_KEY") or secrets.token_urlsafe(32)
entorno["LLAMA_API_KEY"] = clave
log = RUNTIME / "llama-server.log"
with log.open("w") as salida:
    proceso = subprocess.Popen(
        [str(servidor), "-m", str(pesos), "--alias", LLM_MODEL, "--host", "127.0.0.1", "--port", "8080",
         "-ngl", "99", "--split-mode", "layer", "--tensor-split", "1,1", "-c", "16384",
         "--parallel", "1", "--jinja", "--flash-attn", "on", "-b", "512", "-ub", "256",
         "--no-webui", "--log-verbosity", "3", "--log-colors", "off"],
        stdout=salida, stderr=subprocess.STDOUT, env=entorno,
    )

limite = time.monotonic() + 600
while True:
    if proceso.poll() is not None or time.monotonic() >= limite:
        if proceso.poll() is None:
            proceso.terminate()
            proceso.wait(timeout=20)
        raise RuntimeError(log.read_text(errors="replace")[-6500:])
    try:
        if requests.get(LLM_BASE_URL + "/health", timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(2)

cliente = OpenAI(base_url=LLM_BASE_URL, api_key=clave, timeout=120, max_retries=0)
assert any(modelo.id == LLM_MODEL for modelo in cliente.models.list().data)
log_arranque = log.read_text(errors="replace")
for linea in log_arranque.splitlines():
    if "offloaded" in linea or ("CUDA" in linea and "buffer size" in linea):
        print(linea)
(Path(WORK) / "llama-arranque.log").write_text(log_arranque, encoding="utf-8")
print("Endpoint local disponible:", LLM_BASE_URL)
print("VRAM tras carga:", gpu_info())


## Ejecutar el banco de 15 preguntas

Se cotejan las consultas al motor y sus argumentos con el banco, tanto si Python elige
la consulta directamente como si la solicita Mistral. El verificador numérico comprueba
que las cifras aparezcan en la pregunta o en las herramientas con la tolerancia solicitada.
La corrección semántica de las tres respuestas especiales se confirma leyendo la salida.
`tokens_s` es el rendimiento efectivo de generación, incluyendo los turnos de herramientas.

In [ ]:
# (categoría, pregunta, llamadas esperadas[, alternativas válidas])
CASOS = [
    ("directa", "¿Qué disponibilidad propia tiene ST-2390?", [("kpis_etapa", {"nombre": "ST2390"})]),
    ("directa", "¿Cuántas horas de parada propia acumula ST0050?", [("kpis_etapa", {"nombre": "ST0050"})]),
    ("directa", "¿Cuántas etapas y días cubre el análisis de la línea?", [("resumen_linea", {})]),
    ("ranking", "¿Qué 3 máquinas penalizan más la línea?", [("prioridades", {"n": 3})]),
    ("ranking", "¿Cuál es el cuello de botella principal y a qué zona de la línea afecta?",
     [("cuello_botella", {})]),
    ("ranking", "¿A qué 3 máquinas mando primero a mantenimiento?", [("mantenimiento", {"n": 3})]),
    ("comparación", "Compara la disponibilidad propia de ST-0050 y ST-2390.",
     [("kpis_etapa", {"nombre": "ST0050"}), ("kpis_etapa", {"nombre": "ST2390"})]),
    ("comparación", "Compara las horas de parada inducida de ST_0020 y ST-0050.",
     [("kpis_etapa", {"nombre": "ST0020"}), ("kpis_etapa", {"nombre": "ST0050"})]),
    ("concepto", "¿Qué es la parada inducida?", [("glosario", {"termino": "TAI"})]),
    ("concepto", "¿Qué es Dp y cuál es su objetivo?", [("glosario", {"termino": "Dp"})]),
    ("fiabilidad", "¿Me puedo fiar de los datos de ST2235?", [("alertas", {"nombre": "ST2235"})],
     [[("kpis_etapa", {"nombre": "ST2235"})],
      [("kpis_etapa", {"nombre": "ST2235"}), ("alertas", {"nombre": "ST2235"})]]),
    ("fiabilidad", "¿Qué alertas afectan a la calidad del registro de PM_0110?",
     [("alertas", {"nombre": "PM0110"})]),
    ("sin datos", "¿Qué temperatura tuvo el motor de ST-2390 en grados Celsius?", [],
     [[("kpis_etapa", {"nombre": "ST2390"})]]),
    ("sin datos", "¿Cuánto cuesta una hora de parada de la línea en euros?", []),
    ("premisa falsa", "¿Por qué ST_0030 es la que más produce?", [("alertas", {"nombre": "ST0030"})],
     [[("kpis_etapa", {"nombre": "ST0030"})]]),
]

def llamadas_correctas(llamadas, esperadas, alternativas=()):
    firma = firma_llamadas(llamadas)
    return any(firma == firma_llamadas(opcion) for opcion in [esperadas, *alternativas])

def juicio_preliminar(categoria, respuesta):
    texto = respuesta.casefold()
    if categoria == "sin datos":
        return bool(re.search(r"no (?:hay|tengo|dispongo|se dispone|se mide|se registr|puedo|está|se puede|consta)|sin datos", texto))
    if categoria == "premisa falsa":
        return bool(re.search(r"sin datos|no (?:hay|registra|tiene|dispongo|se dispone|se puede|puedo)|excluida", texto))
    return None

def evaluar(casos):
    inicio = gpu_info()
    maximos = [gpu[2] for gpu in inicio]
    detener = threading.Event()
    def medir():
        while not detener.wait(0.5):
            for i, (_, _, usada) in enumerate(gpu_info()):
                maximos[i] = max(maximos[i], usada)
    monitor = threading.Thread(target=medir, daemon=True)
    monitor.start()
    filas = []
    try:
        for i, (categoria, pregunta, esperadas, *alternativas) in enumerate(casos, 1):
            comienzo = time.perf_counter()
            respuesta, llamadas, resultados, tokens = preguntar(cliente, LLM_MODEL, herramientas, pregunta)
            latencia = time.perf_counter() - comienzo
            cifras = verificar_cifras(respuesta, pregunta, resultados)
            filas.append({
                "modelo": LLM_MODEL, "caso": i, "categoria": categoria, "pregunta": pregunta,
                "respuesta": respuesta, "llamadas_esperadas": json.dumps(esperadas, ensure_ascii=False),
                "llamadas_reales": json.dumps(llamadas, ensure_ascii=False),
                "resultados_herramientas": json.dumps(resultados, ensure_ascii=False),
                "herramientas_correctas": llamadas_correctas(llamadas, esperadas, *alternativas),
                "cifras_verificadas": not cifras, "cifras_no_verificadas": json.dumps(cifras),
                "juicio_preliminar_sin_dato_o_premisa": juicio_preliminar(categoria, respuesta),
                "latencia_s": latencia, "tokens_generados": tokens, "tokens_s": tokens / latencia,
            })
            print(f"{i:02d}/{len(casos)} · {categoria} · {latencia:.1f} s", flush=True)
    finally:
        detener.set()
        monitor.join()
    return pd.DataFrame(filas).assign(vram_t4_0_mib=maximos[0], vram_t4_1_mib=maximos[1])

resultados = evaluar(CASOS)
destino = WORK / "Datos_modelo" / "llm_benchmark.csv"
destino.parent.mkdir(exist_ok=True)
resultados.to_csv(destino, index=False)


## Resultados de la prueba

In [ ]:
tabla = resultados.groupby("modelo").agg(
    preguntas=("caso", "count"), cifras_verificadas=("cifras_verificadas", "mean"),
    herramientas_correctas=("herramientas_correctas", "mean"),
    sin_dato_o_premisa_preliminar=("juicio_preliminar_sin_dato_o_premisa", "mean"),
    latencia_media_s=("latencia_s", "mean"), tokens_s=("tokens_s", "mean"),
    vram_t4_0_mib=("vram_t4_0_mib", "max"), vram_t4_1_mib=("vram_t4_1_mib", "max"),
)
for columna in ("cifras_verificadas", "herramientas_correctas", "sin_dato_o_premisa_preliminar"):
    tabla[columna] *= 100
display(tabla.round(2))
display(resultados.loc[resultados.categoria.isin(["sin datos", "premisa falsa"]),
                       ["caso", "pregunta", "respuesta", "juicio_preliminar_sin_dato_o_premisa"]])


## Validar el prompt con preguntas nuevas

Revisa cada respuesta contra sus herramientas. Los casos comprueban unidades, atribución
de cifras a su etapa, operaciones excluidas o inexistentes y datos fuera del alcance.
El verificador automático comprueba valores, y la revisión comprueba también su significado.

In [ ]:
CASOS_VALIDACION = [
    ("unidad", "¿Cuál es el tiempo de ciclo de PM-0110? Dame el valor con la unidad original.",
     [("kpis_etapa", {"nombre": "PM0110"})]),
    ("etapa paralela", "¿Qué disponibilidad propia tiene ST-2375? ¿Es de esa máquina sola?",
     [("kpis_etapa", {"nombre": "ST2375"})]),
    ("sin datos", "¿En qué turno de la noche se averió ST-2390?",
     [], [[("kpis_etapa", {"nombre": "ST2390"})], [("alertas", {"nombre": "ST2390"})]]),
    ("premisa falsa", "ST_0030 hizo cero piezas, ¿verdad? ¿Fue una avería del sensor?",
     [("alertas", {"nombre": "ST0030"})], [[("kpis_etapa", {"nombre": "ST0030"})],
      [("kpis_etapa", {"nombre": "ST0030"}), ("alertas", {"nombre": "ST0030"})]]),
    ("máquina desconocida", "¿Cuántas piezas registró ST-9999?",
     [("kpis_etapa", {"nombre": "ST9999"})]),
    ("comparación", "¿Qué etapa tiene mayor disponibilidad propia, PM-0110 o ST-2390? Dame ambas cifras.",
     [("kpis_etapa", {"nombre": "PM0110"}), ("kpis_etapa", {"nombre": "ST2390"})]),
    ("concepto", "¿Qué significa saturación (SAT) en esta línea?",
     [("glosario", {"termino": "SAT"})]),
    ("ranking", "Dime las tres primeras etapas del ranking general de prioridades.",
     [("prioridades", {"n": 3})]),
]
validacion = evaluar(CASOS_VALIDACION)
validacion.to_csv(destino.with_name("llm_validacion_prompt.csv"), index=False)
display(validacion[["caso", "categoria", "respuesta", "herramientas_correctas", "cifras_verificadas"]])

## Hacer una consulta durante la sesión

Ejecuta la celda cambiando la pregunta. La API sigue en `127.0.0.1:8080/v1` mientras viva
la sesión interactiva. Consulta `chat/README.md` para conectar un cliente externo.

In [ ]:
from chat.herramientas import responder

if os.environ.get("HORSE_DEMO_PREGUNTA"):
    salida = responder(cliente, LLM_MODEL, herramientas, os.environ["HORSE_DEMO_PREGUNTA"])
    print(salida["respuesta"])
    print()
    if salida["sin_datos"]:
        print("⚠️ El modelo respondió sin consultar ninguna herramienta.")
    if salida["verificada"]:
        print("✅ Cifras verificadas")
    else:
        print("⚠️ Revisar, no se pueden respaldar:", ", ".join(salida["cifras_no_verificadas"]))
